In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MU = 1.0  # nonlinearity factor
X0 = 1.0  # initial position x(0)
V0 = 0.0  # initial velocity x'(0)
WINDOW_SIZE = 2.0  # length of each time-marching window

PERIOD_MARGIN = 1.15
ROUGH_PERIOD_FACTOR = 1.614
N_PERIODS_SETTLE = 4

LAYERS = (1, 32, 32, 32, 32, 1)
N_COLLOCATION_PER_WINDOW = 2000
ADAM_EPOCHS = 3000
ADAM_LR = 1e-3
LBFGS_MAX_ITER = 2000
PRINT_EVERY = 500

FOURIER_MAPPING_SIZE = 12
FOURIER_SIGMA = 1.0

# Fourier-specific: joint sigma / mapping-size sweep
SIGMA_CANDIDATES = [0.5, 1, 2, 5, 10, 20, 30]
MAPPING_SIZE_CANDIDATES = [12, 16, 32]
SIGMA_PROBE_EPOCHS = 800

In [ ]:
def van_der_pol_rhs(t, y, mu=MU):
    x, v = y
    return [v, mu * (1 - x ** 2) * v - x]

def estimate_vdp_period(mu, x0=X0, v0=V0, rough_factor=ROUGH_PERIOD_FACTOR, n_settle=N_PERIODS_SETTLE):
    rough_T = rough_factor * mu
    t_span = n_settle * max(rough_T, 5.0)
    def event_peak(t, y):
        return y[1]
    event_peak.direction = -1

    sol = solve_ivp(
        lambda t, y: van_der_pol_rhs(t, y, mu=mu),
        [0.0, t_span], [x0, v0],
        events=event_peak, rtol=1e-10, atol=1e-12, max_step=0.05,
    )
    peak_times = sol.t_events[0]
    if len(peak_times) < 3:
        print(f"[period detection] not enough peaks found, falling back to rough estimate {rough_T:.3f}")
        return rough_T
    period = peak_times[-1] - peak_times[-2]
    return period

vdp_period = estimate_vdp_period(MU)
T_FINAL = float(PERIOD_MARGIN * vdp_period)
print(f"Detected VdP period at mu={MU}: {vdp_period:.4f}  ->  T_FINAL = {T_FINAL:.4f}")

N_WINDOWS = int(round(T_FINAL / WINDOW_SIZE))
window_edges = np.array([w * WINDOW_SIZE for w in range(N_WINDOWS + 1)])
window_edges[-1] = T_FINAL  # snap the last edge exactly to T_FINAL in case of rounding
window_widths = np.diff(window_edges)
print(f"Uniform window edges (t): {np.round(window_edges, 3)}")
print(f"Window widths: {np.round(window_widths, 3)}")
print(f"Window count: {N_WINDOWS}")

In [ ]:
class BasePINN(nn.Module):
    # Shared architecture skeleton: encode_input() + trunk_forward()

    def __init__(self):
        super().__init__()

    def encode_input(self, t):
        raise NotImplementedError

    def trunk_forward(self, encoded):
        raise NotImplementedError

    def forward(self, t):
        return self.trunk_forward(self.encode_input(t))

    @staticmethod
    def init_linear(linear):
        nn.init.xavier_normal_(linear.weight)
        nn.init.zeros_(linear.bias)

In [ ]:
class FourierFeaturePINN(BasePINN):
    """
    Random Fourier Feature PINN (Tancik et al., "Fourier Features Let Networks
    Learn High Frequency Functions in Low Dimensional Domains").

    encode_input():1. Normalize t into the window-local range [-1, 1], exactly like VanillaPINN.
    2. Map that scalar through a fixed (non-trainable) random frequency matrix B,
    producing gamma(t) = [cos(2*pi*B*t), sin(2*pi*B*t)].
    This lifts the 1-D time input into a 2*mapping_size-D feature vector
    before it ever reaches the MLP, which is what lets the network fit
    the fast oscillations of a stiff/high-mu Van der Pol solution more
    easily than a plain tanh MLP fed a raw scalar.
    trunk_forward():
        Same tanh MLP shape as VanillaPINN, just with an input layer sized for
        2*mapping_size instead of 1.
    """

    def __init__(self, t_start, window_size, layers=LAYERS,
                 mapping_size=FOURIER_MAPPING_SIZE, sigma=FOURIER_SIGMA, B=None):
        super().__init__()
        self.t_start = t_start
        self.window_size = window_size
        self.mapping_size = mapping_size
        self.sigma = sigma

        if B is None:
            B = torch.randn(1, mapping_size) * sigma
        else:
            B = B.clone().detach()
        self.register_buffer("B", B)


        # Trunk: input dim is 2*mapping_size (cos + sin features) instead of layers[0].
        # Hidden/output widths are taken from `layers[1:]` so LAYERS stays shared
        # config, only the input width changes to match the Fourier embedding.
        fourier_layers = (2 * mapping_size,) + tuple(layers[1:])
        self.trunk = nn.ModuleList()
        for i in range(len(fourier_layers) - 1):
            linear = nn.Linear(fourier_layers[i], fourier_layers[i + 1])
            self.init_linear(linear)
            self.trunk.append(linear)
        self.activation = torch.tanh

    def encode_input(self, t):
        # Same local normalization as VanillaPINN: tau in [-1, 1] over this window.
        tau = t - self.t_start
        t_norm = (2.0 * tau / self.window_size) - 1.0
        # Random Fourier feature mapping: (N, 1) @ (1, mapping_size) -> (N, mapping_size)
        proj = 2.0 * np.pi * (t_norm @ self.B)
        return torch.cat([torch.cos(proj), torch.sin(proj)], dim=-1)

    def trunk_forward(self, encoded):
        out = encoded
        for layer in self.trunk[:-1]:
            out = self.activation(layer(out))
        return self.trunk[-1](out)

In [ ]:
class WindowPINN(nn.Module):
    """
    Time-marching wrapper, wraps base architecture (FourierFeaturePinn) and enforces this window's initial condition exactly:
    x(t) = x0_local + v0_local * tau + tau^2 * base_model(t)
    """
    def __init__(self, base_model, t_start, window_size, x0_local, v0_local):
        super().__init__()
        self.base_model = base_model
        self.t_start = t_start
        self.window_size = window_size
        self.x0_local = x0_local
        self.v0_local = v0_local

    def forward(self, t):
        tau = t - self.t_start
        n_raw = self.base_model(t)
        return self.x0_local + self.v0_local * tau + (tau ** 2) * n_raw

In [ ]:
def physics_residual(model, t, mu=MU):
    x = model(t)
    x_t = torch.autograd.grad(x, t, grad_outputs=torch.ones_like(x), create_graph=True, retain_graph=True)[0]
    x_tt = torch.autograd.grad(x_t, t, grad_outputs=torch.ones_like(x_t), create_graph=True, retain_graph=True)[0]
    return x_tt - mu * (1.0 - x ** 2) * x_t + x

def compute_loss(model, t_f):
    # only the physics residual; IC is hard-enforced, no loss_ic term
    f_pred = physics_residual(model, t_f)
    return torch.mean(f_pred ** 2)

def sample_collocation_points(t_start, t_end, n):
    t_f = np.random.uniform(low=t_start, high=t_end, size=(n, 1))
    t_f = torch.tensor(t_f, dtype=torch.get_default_dtype(), device=device)
    t_f.requires_grad_(True)
    return t_f

def evaluate_state(model, t_value):
    # (x, x') of a trained window model at a single time becomes the hard IC handed to the next window
    t_tensor = torch.tensor([[t_value]], dtype=torch.get_default_dtype(), device=device, requires_grad=True)
    x = model(t_tensor)
    x_t = torch.autograd.grad(x, t_tensor, grad_outputs=torch.ones_like(x))[0]
    return x.item(), x_t.item()

In [ ]:
def quick_pinn_probe(sigma, mapping_size, t_f, t_start, window_size, adam_epochs=SIGMA_PROBE_EPOCHS):
    torch.manual_seed(SEED)
    base = FourierFeaturePINN(t_start, window_size, mapping_size=mapping_size, sigma=sigma).to(device)
    model_w = WindowPINN(base, t_start, window_size, X0, V0).to(device)
    opt = torch.optim.Adam(model_w.parameters(), lr=ADAM_LR)
    loss = None
    for _ in range(adam_epochs):
        opt.zero_grad()
        loss = compute_loss(model_w, t_f)
        loss.backward()
        opt.step()
    return loss.item()

In [ ]:
def train_window(model, t_f, adam_epochs=ADAM_EPOCHS, adam_lr=ADAM_LR, lbfgs_max_iter=LBFGS_MAX_ITER, print_every=PRINT_EVERY, window_idx=0):
    history = []
    optimizer_adam = torch.optim.Adam(model.parameters(), lr=adam_lr)
    for epoch in range(adam_epochs):
        optimizer_adam.zero_grad()
        loss = compute_loss(model, t_f)
        loss.backward()
        optimizer_adam.step()
        history.append(loss.item())
        if epoch % print_every == 0:
            print(f"[window {window_idx}] [Adam]  epoch {epoch:5d} | loss {loss.item():.3e}")
    optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=lbfgs_max_iter, tolerance_grad=1e-9, tolerance_change=1e-12, history_size=50, line_search_fn="strong_wolfe")
    counter = {"n": 0}
    def closure():
        optimizer_lbfgs.zero_grad()
        loss = compute_loss(model, t_f)
        loss.backward()
        history.append(loss.item())
        counter["n"] += 1
        if counter["n"] % print_every == 0:
            print(f"[window {window_idx}] [L-BFGS] iter {counter['n']:5d} | loss {loss.item():.3e}")
        return loss

    optimizer_lbfgs.step(closure)
    return history

In [ ]:
print("Joint sigma / mapping-size sweep (multiple representative windows)")
probe_indices = sorted(set([0, N_WINDOWS // 2, N_WINDOWS - 1]))
probe_t_fs = {
    w: sample_collocation_points(window_edges[w], window_edges[w + 1], N_COLLOCATION_PER_WINDOW)
    for w in probe_indices
}

combo_losses = {}
for mapping_size_candidate in MAPPING_SIZE_CANDIDATES:
    for sigma_candidate in SIGMA_CANDIDATES:
        window_losses = [
            quick_pinn_probe(sigma_candidate, mapping_size_candidate, probe_t_fs[w],
                              t_start=float(window_edges[w]), window_size=float(window_widths[w]))
            for w in probe_indices
        ]
        avg_loss = float(np.mean(window_losses))
        combo_losses[(sigma_candidate, mapping_size_candidate)] = avg_loss
        per_window_str = ", ".join(f"{l:.2e}" for l in window_losses)
        print(f"  sigma={sigma_candidate:>5} mapping_size={mapping_size_candidate:>3} "
              f"-> avg probe loss {avg_loss:.3e}  (per-window: {per_window_str})")

FOURIER_SIGMA, FOURIER_MAPPING_SIZE = min(combo_losses, key=combo_losses.get)
print(f"- Selected FOURIER_SIGMA = {FOURIER_SIGMA}, FOURIER_MAPPING_SIZE = {FOURIER_MAPPING_SIZE} -\n")

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

start_time = time.time()

models = []
full_history = []
x0_cur, v0_cur = X0, V0

for w in range(N_WINDOWS):
    t_start = float(window_edges[w])
    t_end = float(window_edges[w + 1])
    width = float(window_widths[w])

    t_f = sample_collocation_points(t_start, t_end, N_COLLOCATION_PER_WINDOW)
    base = FourierFeaturePINN(t_start, width, mapping_size=FOURIER_MAPPING_SIZE, sigma=FOURIER_SIGMA).to(device)
    model_w = WindowPINN(base, t_start, width, x0_cur, v0_cur).to(device)

    hist = train_window(model_w, t_f, window_idx=w)
    full_history.extend(hist)

    x0_cur, v0_cur = evaluate_state(model_w, t_end)
    models.append(model_w)

training_time = time.time() - start_time
print(f"Time-marching training complete in {training_time:.1f}s.")

In [ ]:
t_eval = np.linspace(0, T_FINAL, 2000)
sol = solve_ivp(van_der_pol_rhs, [0, T_FINAL], [X0, V0], t_eval=t_eval, method="RK45", rtol=1e-10, atol=1e-10)
x_reference = sol.y[0]
v_reference = sol.y[1]

x_pred = np.zeros_like(t_eval)
for w, model_w in enumerate(models):
    t_start = window_edges[w]
    t_end = window_edges[w + 1]
    mask = (t_eval >= t_start) & (t_eval <= t_end + 1e-9)
    if not np.any(mask):
        continue
    t_local = torch.tensor(t_eval[mask].reshape(-1, 1), dtype=torch.get_default_dtype(), device=device)
    with torch.no_grad():
        x_local = model_w(t_local).cpu().numpy().flatten()
    x_pred[mask] = x_local

In [ ]:
mse = np.mean((x_pred - x_reference) ** 2)
relative_l2_error = np.linalg.norm(x_pred - x_reference) / np.linalg.norm(x_reference)
final_training_loss = full_history[-1]
n_params = sum(p.numel() for p in models[0].parameters())

# Calculate additional metrics
mae = np.mean(np.abs(x_pred - x_reference))
max_error = np.max(np.abs(x_pred - x_reference))
rmse = np.sqrt(mse)

# physics residual evaluated on a fresh set of test points across the whole domain
t_res_test = torch.tensor(
    np.random.uniform(0.0, T_FINAL, size=(2000, 1)),
    dtype=torch.get_default_dtype(), device=device,
)
residual_vals = []
for w, model_w in enumerate(models):
    t_start = window_edges[w]
    t_end = window_edges[w + 1]
    mask = (t_res_test.detach().cpu().numpy().flatten() >= t_start) & \
           (t_res_test.detach().cpu().numpy().flatten() <= t_end)
    if not np.any(mask):
        continue
    t_sub = t_res_test[mask].clone().detach().requires_grad_(True)
    f_sub = physics_residual(model_w, t_sub)
    residual_vals.append(f_sub.detach().cpu().numpy())
mean_physics_residual = np.mean(np.concatenate(residual_vals) ** 2) if residual_vals else float("nan")

print(f"Selected sigma: {FOURIER_SIGMA}")
print(f"Selected mapping_size: {FOURIER_MAPPING_SIZE}")
print(f"MSE: {mse:.3e}")
print(f"Relative L2 error: {relative_l2_error:.3e}")
print(f"Final training loss: {final_training_loss:.3e}")
print(f"Mean physics residual (test points): {mean_physics_residual:.3e}")
print(f"Training time: {training_time:.1f}s")
print(f"Trainable parameters (per window model): {n_params}")
print(f"MAE: {mae:.3e}")
print(f"Max Error (L_inf): {max_error:.3e}")
print(f"RMSE: {rmse:.3e}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(t_eval, x_reference, "b-", linewidth=2, label="Reference (RK45)")
axes[0, 0].plot(t_eval, x_pred, "r--", linewidth=2, label="Fourier Feature PINN (time-marched)")
for w in range(1, N_WINDOWS):
    axes[0, 0].axvline(w * WINDOW_SIZE, color="gray", linestyle=":", linewidth=0.7)
axes[0, 0].set_xlabel("t")
axes[0, 0].set_ylabel("x(t)")
axes[0, 0].set_title("Van der Pol oscillator: Fourier Feature PINN vs. reference")
axes[0, 0].legend()

axes[0, 1].semilogy(full_history, label="physics residual loss")
axes[0, 1].set_xlabel("iteration (concatenated across windows)")
axes[0, 1].set_ylabel("loss (log scale)")
axes[0, 1].set_title("Training loss history")
axes[0, 1].legend()

axes[1, 0].plot(t_eval, np.abs(x_pred - x_reference), "k-")
axes[1, 0].set_xlabel("t")
axes[1, 0].set_ylabel("|x_pred - x_reference|")
axes[1, 0].set_title("Absolute error")

x_pred_dot = np.gradient(x_pred, t_eval)
axes[1, 1].plot(x_reference, v_reference, "b-", linewidth=2, label="Reference")
axes[1, 1].plot(x_pred, x_pred_dot, "r--", linewidth=2, label="Fourier Feature PINN")
axes[1, 1].set_xlabel("x")
axes[1, 1].set_ylabel("x'")
axes[1, 1].set_title("Phase portrait")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("fourier_pinn_van_der_pol_results.png", dpi=150)
plt.show()